In [10]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [11]:
# Load FCC Cable dataset
fcc_path = "/Users/priyankaramachandran/Desktop/UCD Courses/CN Project/dataset/FCC Dataset.csv"

fcc = pd.read_csv(fcc_path)
fcc.head()

,frn,provider_id,brand_name,location_id,technology,max_advertised_download_speed,max_advertised_upload_speed,low_latency,business_residential_code,state_usps,block_geoid,h3_res8_id
0,25646373,130235,Spectrum,1368830548,40,1000,35,1,X,CA,60590886022008,8829a0b443fffff
1,25646373,130235,Spectrum,1368830977,40,1000,35,1,X,CA,60590218163003,8829a0a2a5fffff
2,3768165,130317,Xfinity,1319686581,40,1200,35,1,X,CA,60855027041016,8828340983fffff
3,3768165,130317,Xfinity,1344590546,40,2000,250,1,X,CA,60816072002010,882834298dfffff
4,3768165,130317,Xfinity,1344635958,40,2000,250,1,X,CA,60816072002012,882834298dfffff


In [12]:
# Standardize column names
fcc.columns = fcc.columns.str.lower().str.strip()
fcc.columns

Index(['frn', 'provider_id', 'brand_name', 'location_id', 'technology',
       'max_advertised_download_speed', 'max_advertised_upload_speed',
       'low_latency', 'business_residential_code', 'state_usps', 'block_geoid',
       'h3_res8_id'],
      dtype='object')

In [13]:
keep_cols = [
    "frn",
    "provider_id",
    "brand_name",
    "technology",
    "max_advertised_download_speed",
    "max_advertised_upload_speed",
    "low_latency",
    "business_residential_code",
    "state_usps",
    "block_geoid"
]

fcc = fcc[keep_cols].copy()
fcc.head()

,frn,provider_id,brand_name,technology,max_advertised_download_speed,max_advertised_upload_speed,low_latency,business_residential_code,state_usps,block_geoid
0,25646373,130235,Spectrum,40,1000,35,1,X,CA,60590886022008
1,25646373,130235,Spectrum,40,1000,35,1,X,CA,60590218163003
2,3768165,130317,Xfinity,40,1200,35,1,X,CA,60855027041016
3,3768165,130317,Xfinity,40,2000,250,1,X,CA,60816072002010
4,3768165,130317,Xfinity,40,2000,250,1,X,CA,60816072002012


In [14]:
# Convert datatypes
fcc["brand_name"] = fcc["brand_name"].astype(str).str.strip()
fcc["state_usps"] = fcc["state_usps"].astype(str).str.strip()
fcc["business_residential_code"] = fcc["business_residential_code"].astype(str).str.strip()

fcc["max_advertised_download_speed"] = pd.to_numeric(
    fcc["max_advertised_download_speed"], errors="coerce"
)

fcc["max_advertised_upload_speed"] = pd.to_numeric(
    fcc["max_advertised_upload_speed"], errors="coerce"
)

In [15]:
# Clean block GEOID
fcc["block_geoid"] = (
    fcc["block_geoid"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

fcc["block_geoid"].head()

0    60590886022008
1    60590218163003
2    60855027041016
3    60816072002010
4    60816072002012
Name: block_geoid, dtype: object

In [16]:
# If block_geoid appears in scientific notation, reload CSV using dtype
fcc = pd.read_csv(fcc_path, dtype={"block_geoid": str})

fcc.columns = fcc.columns.str.lower().str.strip()

fcc = fcc[keep_cols].copy()

fcc["block_geoid"] = fcc["block_geoid"].astype(str).str.strip()
fcc["block_geoid"].head()

0    060590886022008
1    060590218163003
2    060855027041016
3    060816072002010
4    060816072002012
Name: block_geoid, dtype: object

In [17]:
# Filter California
fcc = fcc[fcc["state_usps"] == "CA"].copy()

fcc.shape

(9391601, 10)

In [18]:
# Filter Sacramento County
# Sacramento County GEOID starts with 06067

fcc = fcc[fcc["block_geoid"].str.startswith("06067")].copy()

fcc.shape

(456093, 10)

In [19]:
fcc.head()

,frn,provider_id,brand_name,technology,max_advertised_download_speed,max_advertised_upload_speed,low_latency,business_residential_code,state_usps,block_geoid
17,3768165,130317,Xfinity,40,1200,35,1,X,CA,060670052051023
58,3768165,130317,Xfinity,40,2000,250,1,X,CA,060670033002024
89,3768165,130317,Xfinity,40,2000,250,1,X,CA,060670033002020
98,3768165,130317,Xfinity,40,1200,35,1,X,CA,060670093312009
104,3768165,130317,Xfinity,40,1200,35,1,X,CA,060670024003002


In [20]:
# Remove missing speed values
before_rows = len(fcc)

fcc = fcc.dropna(subset=[
    "max_advertised_download_speed",
    "max_advertised_upload_speed",
    "block_geoid",
    "brand_name"
]).copy()

print("Rows before:", before_rows)
print("Rows after:", len(fcc))
print("Rows removed:", before_rows - len(fcc))

Rows before: 456093
Rows after: 456093
Rows removed: 0


In [21]:
# Create tract GEOID from block GEOID
# Census block GEOID = 15 digits
# Census tract GEOID = first 11 digits

fcc["tract_geoid"] = fcc["block_geoid"].str[:11]

fcc[["block_geoid", "tract_geoid"]].head()

,block_geoid,tract_geoid
17,060670052051023,06067005205
58,060670033002024,06067003300
89,060670033002020,06067003300
98,060670093312009,06067009331
104,060670024003002,06067002400


Saved cleaned FCC data to cleaned_dataset/fcc_sacramento_cleaned.csv
